In [3]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=20, n_informative=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

num_wolves = 10
max_iter = 20
num_features = X.shape[1]

wolves = np.random.uniform(0, 1, (num_wolves, num_features))

alpha_pos, alpha_score = np.zeros(num_features), float('inf')
beta_pos,  beta_score  = np.zeros(num_features), float('inf')
delta_pos, delta_score = np.zeros(num_features), float('inf')

for it in range(max_iter):
    for i in range(num_wolves):
        selected = np.where(wolves[i] > 0.5)[0]  
        if len(selected) == 0:
            fitness = 1.0
        else:
            model = GaussianNB().fit(X_train[:, selected], y_train)
            error = 1.0 - accuracy_score(y_test, model.predict(X_test[:, selected]))
            penalty = 0.01 * (len(selected) / num_features)  # Prefer fewer features
            fitness = error + penalty

        if fitness < alpha_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score, beta_pos = alpha_score, alpha_pos.copy()
            alpha_score, alpha_pos = fitness, wolves[i].copy()
        elif fitness < beta_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score, beta_pos = fitness, wolves[i].copy()
        elif fitness < delta_score:
            delta_score, delta_pos = fitness, wolves[i].copy()
            
    a = 2.0 - it * (2.0 / max_iter)

    for i in range(num_wolves):
        for d in range(num_features):
            X1 = alpha_pos[d] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * alpha_pos[d] - wolves[i, d])
            X2 = beta_pos[d]  - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * beta_pos[d]  - wolves[i, d])
            X3 = delta_pos[d] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * delta_pos[d] - wolves[i, d])
            wolves[i, d] = np.clip((X1 + X2 + X3) / 3.0, 0, 1)

best_features = np.where(alpha_pos > 0.5)[0]

base_acc = accuracy_score(y_test, GaussianNB().fit(X_train, y_train).predict(X_test))
gwo_acc = accuracy_score(y_test, GaussianNB().fit(X_train[:, best_features], y_train).predict(X_test[:, best_features]))

print(f"Baseline Accuracy (All 20 Features): {base_acc * 100:.2f}%")
print(f"GWO Spam Filter Accuracy: {gwo_acc * 100:.2f}%")
print(f"Selected Features ({len(best_features)}/20): {list(best_features)}")

Baseline Accuracy (All 20 Features): 86.33%
GWO Spam Filter Accuracy: 90.67%
Selected Features (6/20): [4, 9, 10, 12, 13, 18]
